# Olist E-Commerce EDA
브라질 이커머스 수요 예측 프로젝트 — 탐색적 데이터 분석

## 1. 라이브러리 불러오기

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

## 2. 데이터 불러오기

In [ ]:
path = "../data/"  # csv 경로 수정

orders      = pd.read_csv(path + "olist_orders_dataset.csv")
order_items = pd.read_csv(path + "olist_order_items_dataset.csv")
products    = pd.read_csv(path + "olist_products_dataset.csv")
customers   = pd.read_csv(path + "olist_customers_dataset.csv")
sellers     = pd.read_csv(path + "olist_sellers_dataset.csv")
payments    = pd.read_csv(path + "olist_order_payments_dataset.csv")
reviews     = pd.read_csv(path + "olist_order_reviews_dataset.csv")

## 3. 데이터 기본 구조 확인

In [ ]:
dfs = {
    "orders":      orders,
    "order_items": order_items,
    "products":    products,
    "customers":   customers,
    "sellers":     sellers,
    "payments":    payments,
    "reviews":     reviews,
}

for name, df in dfs.items():
    print(f"\n{'='*16} {name} {'='*16}")
    print("\n[ HEAD ]")
    print(df.head(2))
    print("\n[ INFO ]")
    df.info()
    print("\n[ NULL ]")
    print(df.isnull().sum())
    print("\n[ SHAPE ]", df.shape)

## 4. 날짜 데이터 처리

> **수정** `.dt.date` → `.dt.normalize()`  
> `.dt.date`는 Python `date` 객체를 반환해 이후 `reindex`, `resample`, lag feature 계산에서 오류가 납니다.  
> `.dt.normalize()`는 시간을 00:00:00으로 맞춰 `datetime64` 타입을 그대로 유지합니다.

In [ ]:
orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp']
)

# [수정] dt.date → dt.normalize() : datetime64 타입 유지
orders['order_date'] = orders['order_purchase_timestamp'].dt.normalize()

print("order_date dtype:", orders['order_date'].dtype)  # datetime64[ns] 확인

## 5. 주문 기간 확인 및 분석 범위 결정

> **추가** 2016년 데이터는 수집 초기라 주문량이 매우 적어 시계열 모델에 노이즈가 됩니다.  
> 분석 기간을 **2017-01-01 ~ 2018-08-31** 로 명시적으로 제한합니다.

In [ ]:
print("전체 기간")
print("  START:", orders['order_purchase_timestamp'].min())
print("  END:  ", orders['order_purchase_timestamp'].max())

# [추가] 분석 기간 필터링
ANALYSIS_START = '2017-01-01'
ANALYSIS_END   = '2018-08-31'

orders = orders[
    (orders['order_date'] >= ANALYSIS_START) &
    (orders['order_date'] <= ANALYSIS_END)
]

print(f"\n분석 기간 필터링 후")
print("  START:", orders['order_date'].min())
print("  END:  ", orders['order_date'].max())
print("  주문 수:", len(orders))

## 6. 일별 주문량 생성

In [ ]:
daily_orders = (
    orders
    .groupby('order_date')
    .size()
    .reset_index(name='order_count')
)

daily_orders.head()

## 7. 일별 주문량 시각화 + 이상치 탐지

> **추가** 단순 시각화에서 이상치(outlier) 탐지를 추가했습니다.  
> 블랙프라이데이 등 급등 날짜를 미리 파악해야 예측 모델 해석 시 혼선을 막을 수 있습니다.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 8))

# --- 상단: 전체 시계열 ---
ax1 = axes[0]
ax1.plot(daily_orders['order_date'], daily_orders['order_count'], linewidth=0.8)
ax1.set_title("Daily Order Count (2017-01 ~ 2018-08)")
ax1.set_xlabel("Date")
ax1.set_ylabel("Orders")
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax1.tick_params(axis='x', rotation=45)

# [추가] 이상치 탐지: 평균 + 2σ 초과 날짜 표시
mean_val = daily_orders['order_count'].mean()
std_val  = daily_orders['order_count'].std()
threshold = mean_val + 2 * std_val

outliers = daily_orders[daily_orders['order_count'] > threshold]
ax1.axhline(threshold, color='red', linestyle='--', linewidth=0.8, label=f'mean+2σ ({threshold:.0f})')
ax1.scatter(outliers['order_date'], outliers['order_count'], color='red', s=20, zorder=5)
ax1.legend()

# --- 하단: 요일별 평균 주문량 ---
ax2 = axes[1]
daily_orders['weekday'] = daily_orders['order_date'].dt.day_name()
weekday_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
weekday_avg = daily_orders.groupby('weekday')['order_count'].mean().reindex(weekday_order)
ax2.bar(weekday_avg.index, weekday_avg.values)
ax2.set_title("Average Order Count by Weekday")
ax2.set_xlabel("Weekday")
ax2.set_ylabel("Avg Orders")

plt.tight_layout()
plt.show()

print(f"\n이상치 날짜 (평균+2σ = {threshold:.1f} 초과):")
print(outliers[['order_date','order_count']].to_string(index=False))

## 8. 핵심 테이블 Join

> **수정** `price` 컬럼 제거 — 이후 분석에서 미사용이므로 제거해 일관성 확보

In [ ]:
merged = pd.merge(
    orders[['order_id', 'order_date']],
    order_items[['order_id', 'product_id', 'freight_value']],  # [수정] price 제거
    on='order_id',
    how='inner'
)

merged = pd.merge(
    merged,
    products[['product_id', 'product_category_name']],
    on='product_id',
    how='left'
)

print("Join 결과")
print(merged.head())
print("\n데이터 크기:", merged.shape)
print("\n결측치:")
print(merged.isnull().sum())

## 9. 결측치 처리

> **추가** `product_category_name` 결측치를 처리하지 않으면 NaN 카테고리가 demand_df에 그대로 들어갑니다.

In [ ]:
before = len(merged)

# [추가] 카테고리 불명 행 제거
merged = merged.dropna(subset=['product_category_name'])

after = len(merged)
print(f"제거된 행 수: {before - after} ({(before-after)/before*100:.2f}%)")
print(f"남은 행 수: {after}")

## 10. 수요 데이터 생성

In [ ]:
demand_df = (
    merged
    .groupby(['order_date', 'product_category_name'])
    .size()
    .reset_index(name='demand')
)

print("demand_df shape:", demand_df.shape)
demand_df.head()

## 11. 상위 카테고리 추출

> **수정** `products` 기준(재고 수) → `demand_df` 기준(실제 주문량)으로 변경

In [ ]:
TOP_N = 10  # 분석할 카테고리 수

# [수정] products 기준이 아닌 실제 주문량(demand) 기준으로 상위 카테고리 선정
top_categories = (
    demand_df
    .groupby('product_category_name')['demand']
    .sum()
    .sort_values(ascending=False)
    .head(TOP_N)
)

print(f"상위 {TOP_N}개 카테고리 (총 주문량 기준):")
print(top_categories)

# 이후 모델링 루프에서 사용할 리스트
TOP_CATEGORY_LIST = top_categories.index.tolist()
print("\nTOP_CATEGORY_LIST:", TOP_CATEGORY_LIST)

## 12. 날짜 gap 채우기

> **추가** 주문이 없는 날이 누락되어 있으면 lag/rolling feature가 엉뚱한 날짜끼리 계산됩니다.  
> 전체 날짜 범위로 reindex해 0으로 채웁니다.

In [ ]:
full_date_range = pd.date_range(
    start=ANALYSIS_START,
    end=ANALYSIS_END,
    freq='D'
)

filled_frames = []

for cat in TOP_CATEGORY_LIST:
    cat_df = (
        demand_df[demand_df['product_category_name'] == cat]
        .set_index('order_date')[['demand']]
        .reindex(full_date_range, fill_value=0)
    )
    cat_df.index.name = 'order_date'
    cat_df['product_category_name'] = cat
    filled_frames.append(cat_df.reset_index())

demand_filled = pd.concat(filled_frames, ignore_index=True)

print("날짜 gap 채우기 전:", len(demand_df[demand_df['product_category_name'].isin(TOP_CATEGORY_LIST)]))
print("날짜 gap 채우기 후:", len(demand_filled))
print(f"  → 카테고리 {TOP_N}개 × {len(full_date_range)}일 = {TOP_N * len(full_date_range)}행")

## 13. 카테고리별 수요 시계열 시각화

> **수정 + 추가** 단일 카테고리 → 상위 10개 카테고리 subplot으로 확장  
> train/test split 기준선도 함께 표시합니다.

In [ ]:
# train/test 분할 기준 (마지막 60일을 test로)
TRAIN_END = '2018-06-30'
TEST_START = '2018-07-01'

n_cols = 2
n_rows = (TOP_N + 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
axes = axes.flatten()

for i, cat in enumerate(TOP_CATEGORY_LIST):
    ax = axes[i]
    cat_data = demand_filled[demand_filled['product_category_name'] == cat]

    ax.plot(
        cat_data['order_date'],
        cat_data['demand'],
        linewidth=0.8,
        label='demand'
    )

    # train/test 분할선
    ax.axvline(
        pd.Timestamp(TEST_START),
        color='red', linestyle='--', linewidth=0.8, label='test start'
    )

    ax.set_title(cat, fontsize=9)
    ax.set_ylabel("Demand", fontsize=8)
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%y-%m'))

    if i == 0:
        ax.legend(fontsize=7)

# 남는 subplot 제거
for j in range(len(TOP_CATEGORY_LIST), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle(f"Top {TOP_N} Category Demand (train | test split at {TEST_START})", y=1.01)
plt.tight_layout()
plt.show()

## 14. 카테고리별 수요 통계 요약

In [ ]:
summary = (
    demand_filled
    .groupby('product_category_name')['demand']
    .agg(['mean', 'std', 'max', 'min',
          lambda x: (x == 0).mean()])
    .rename(columns={'<lambda_0>': 'zero_rate'})
    .sort_values('mean', ascending=False)
)

summary['cv'] = summary['std'] / summary['mean']  # 변동계수

print("카테고리별 수요 통계 (zero_rate: 주문 0인 날 비율, cv: 변동계수)")
print(summary.round(3).to_string())

## 15. 배송비 분석

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(merged['freight_value'], bins=50)
axes[0].set_title("Freight Value Distribution")
axes[0].set_xlabel("Freight Value (BRL)")
axes[0].set_ylabel("Count")

# 카테고리별 평균 배송비 (상위 10개)
freight_by_cat = (
    merged[merged['product_category_name'].isin(TOP_CATEGORY_LIST)]
    .groupby('product_category_name')['freight_value']
    .mean()
    .sort_values(ascending=True)
)
axes[1].barh(freight_by_cat.index, freight_by_cat.values)
axes[1].set_title("Avg Freight Value by Category (Top 10)")
axes[1].set_xlabel("Avg Freight Value (BRL)")

plt.tight_layout()
plt.show()

print("배송비 통계:")
print(merged['freight_value'].describe())

## 16. Feature Engineering 준비 — 최종 데이터 확인

> 다음 단계(Feature Engineering)로 넘어가기 전 최종 상태를 확인합니다.

In [ ]:
print("=" * 50)
print("Feature Engineering 진입 전 최종 체크")
print("=" * 50)
print(f"분석 기간     : {ANALYSIS_START} ~ {ANALYSIS_END}")
print(f"Train 기간    : {ANALYSIS_START} ~ {TRAIN_END}")
print(f"Test 기간     : {TEST_START} ~ {ANALYSIS_END}")
print(f"대상 카테고리 : {len(TOP_CATEGORY_LIST)}개")
print(f"demand_filled : {demand_filled.shape}")
print(f"order_date dtype : {demand_filled['order_date'].dtype}")
print(f"결측치 : {demand_filled.isnull().sum().sum()}")
print()
print("대상 카테고리 목록:")
for i, cat in enumerate(TOP_CATEGORY_LIST, 1):
    print(f"  {i:2d}. {cat}")
print()
print("demand_filled sample:")
demand_filled.head()